In [1]:
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

In [2]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLIT_DIR = BASE_DIR / "data" / "splits"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "val.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (24266, 11)
Validation: (5200, 11)
Test: (5201, 11)


In [3]:
target_col = "articleType"

classes = sorted(train_df[target_col].unique())
class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

num_classes = len(classes)

print("Number of classes:", num_classes)
print(class_to_idx)

Number of classes: 23
{'Backpacks': 0, 'Belts': 1, 'Briefs': 2, 'Casual Shoes': 3, 'Flats': 4, 'Flip Flops': 5, 'Formal Shoes': 6, 'Handbags': 7, 'Heels': 8, 'Jeans': 9, 'Kurtas': 10, 'Perfume and Body Mist': 11, 'Sandals': 12, 'Shirts': 13, 'Shorts': 14, 'Socks': 15, 'Sports Shoes': 16, 'Sunglasses': 17, 'Tops': 18, 'Trousers': 19, 'Tshirts': 20, 'Wallets': 21, 'Watches': 22}


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
class WardrobeDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = row["image_path"]
        label_name = row["articleType"]
        label = self.class_to_idx[label_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [6]:
train_dataset = WardrobeDataset(train_df, class_to_idx, transform=train_transform)
val_dataset = WardrobeDataset(val_df, class_to_idx, transform=eval_transform)
test_dataset = WardrobeDataset(test_df, class_to_idx, transform=eval_transform)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 24266
Validation dataset: 5200
Test dataset: 5201


In [7]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 759
Validation batches: 163
Test batches: 163


In [8]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([18,  7,  2,  1, 10,  3,  4,  7, 19, 13])


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

images = images.to(device)
labels = labels.to(device)

print("Images device:", images.device)
print("Labels device:", labels.device)

Using device: cuda
Images device: cuda:0
Labels device: cuda:0
